In [1]:
import os
os.environ["TOKENIZERS_PARALLELISM"] = "false"

from qdrant_client import models, qdrant_client
from qdrant_client.models import SparseVector
from fastembed import SparseTextEmbedding
from sentence_transformers import SentenceTransformer

s:\my_projects\Arabic_News_Agentic_RAG\rag_env\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
collection_name = "arabic_news"

# Use relative path from agent directory to the data directory
client = qdrant_client.QdrantClient(path="../data/qdrant_db")

model = SentenceTransformer("aubmindlab/bert-base-arabertv02")
sparse_model = SparseTextEmbedding(model_name="Qdrant/bm25")

C:\Users\saleen\AppData\Local\Temp\ipykernel_23328\3351440819.py:4: UserWarning: Local mode is not recommended for collections with more than 20,000 points. Collection <arabic_news> contains 26320 points. Consider using Qdrant in Docker or Qdrant Cloud for better performance with large datasets.
  client = qdrant_client.QdrantClient(path="../data/qdrant_db")
Loading weights: 100%|██████████| 199/199 [00:00<00:00, 2812.59it/s]
[transformers] BertModel LOAD REPORT from: aubmindlab/bert-base-arabertv02
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 

In [3]:
def _retrieve(query: str, top_k: int =5, category_filter: str = None):
    dense_vec = model.encode(query, normalize_embeddings=True).tolist()
    sparse_vec = list(sparse_model.embed([query]))[0]
    query_filter = None
    if category_filter:
        query_filter = models.Filter(
            must=[models.FieldCondition(
                key="category",
                match=models.MatchValue(value=category_filter)
            )]
        )
    results = client.query_points(
        collection_name=collection_name,
        prefetch=[
            models.Prefetch(query=dense_vec, using="", limit=20, filter=query_filter),
            models.Prefetch(
                query=SparseVector(
                    indices=sparse_vec.indices.tolist(),
                    values=sparse_vec.values.tolist()
                ),
                using="sparse",
                limit=top_k * 2,
                filter=query_filter
            ),
        ],
        query=models.FusionQuery(fusion=models.Fusion.RRF),
        query_filter=query_filter,
        limit=top_k
    )
    return results.points

In [4]:
def search_news(query: str)->dict:
    points =_retrieve(query, top_k=5)
    return {
        "tool":"search_news",
        "query": query,
        "results": [
            {
                "id": str(p.id),
                "category": p.payload["category"],
                "text": p.payload["text"],
                "score": p.score
            }
            for p in points
        ]
    }


In [5]:
def summarize_topic(query: str) -> dict:
    points = _retrieve(query, top_k=5)
    combined_content = " ".join([point.payload["text"] for point in points  ])
    return {
        "tool": "summarize_topic",
        "query": query,
        "context": combined_content,
        "source_count": len(points)
    }



In [6]:
def compare_timeline(query: str, category: str = None ) -> dict:
    points = _retrieve(query, top_k=10,category_filter= category)
    return {
        "tool": "compare_timeline",
        "query" : query,
        "results": [
            {"text":point.payload["text"], "category": point.payload["category"]}
            for point in points
        ]
    }


In [7]:
def answer_direct(query:str) ->dict:
    return {
        "tool": "answer_direct",
        "query": query,
        "context": ""

    }



In [8]:
search_news("ما هي آخر الأخبار السياسية؟")


{'tool': 'search_news',
 'query': 'ما هي آخر الأخبار السياسية؟',
 'results': [{'id': '9f69191c056b445ac619f77501362ec6',
   'category': 'Politics',
   'text': 'ريما مكتبي مع جنبلاط الفراغ الرئاسي في لبنان والاتصالات الجارية للتوصل الى انتخاب رئيس للجمهورية. كما تتناول المقابلة الوضع في سوريا والتوقعات بالنسبة للمرحلة المقبلة، والتاثير على لبنان، اضافة الى ما حمله الى بيروت وزير الخارجية الاميركية جون كيري في زيارته الاخيرة. كما تتطرق المقابلة للتدهور',
   'score': 0.5},
  {'id': '57a9e8f3dbd57bd22c4fd8c92dc6df95',
   'category': 'Politics',
   'text': 'التي ترفعها الحركة هي رفض صمت الرئيس والحكومة والقوى السياسية عما يحدث على الارض. رفض سيطرة الميليشيات المسلحة على العاصمة وعلى بقية المحافظات. رفض الاتفاقيات السياسية مع الميليشيات المسلحة ما لم تفض الى نزع السلاح. رفض اقتحام معسكرات الجيش والامن ونهب السلاح. رفض دمج الميليشيات المسلحة في الجيش',
   'score': 0.5},
  {'id': '9a5e45d014ba6cf741d6b0f29a12c70d',
   'category': 'Politics',
   'text': 'جنبلاط ضيف حصري على قناة الحدث الثلاثاء 

In [9]:
compare_timeline("ما هي آخر الأخبار السياسية؟", category="Politics")


{'tool': 'compare_timeline',
 'query': 'ما هي آخر الأخبار السياسية؟',
 'results': [{'text': 'ريما مكتبي مع جنبلاط الفراغ الرئاسي في لبنان والاتصالات الجارية للتوصل الى انتخاب رئيس للجمهورية. كما تتناول المقابلة الوضع في سوريا والتوقعات بالنسبة للمرحلة المقبلة، والتاثير على لبنان، اضافة الى ما حمله الى بيروت وزير الخارجية الاميركية جون كيري في زيارته الاخيرة. كما تتطرق المقابلة للتدهور',
   'category': 'Politics'},
  {'text': 'التي ترفعها الحركة هي رفض صمت الرئيس والحكومة والقوى السياسية عما يحدث على الارض. رفض سيطرة الميليشيات المسلحة على العاصمة وعلى بقية المحافظات. رفض الاتفاقيات السياسية مع الميليشيات المسلحة ما لم تفض الى نزع السلاح. رفض اقتحام معسكرات الجيش والامن ونهب السلاح. رفض دمج الميليشيات المسلحة في الجيش',
   'category': 'Politics'},
  {'text': 'جنبلاط ضيف حصري على قناة الحدث الثلاثاء الحدث.نت يحل رئيس الحزب الاشتراكي اللبناني وليد جنبلاط غدا الثلاثاء ضيفا على قناة الحدث، في مقابلة حصرية تبث الساعة الخامسة عصرا بتوقيت بيروت، الثانية بعد الظهر بتوقيت غرينتش. وتتناول المقابل

In [10]:
answer_direct("ما هي آخر الأخبار السياسية؟")


{'tool': 'answer_direct',
 'query': 'ما هي آخر الأخبار السياسية؟',
 'context': ''}

In [11]:
summarize_topic("ما هي آخر الأخبار السياسية؟")

{'tool': 'summarize_topic',
 'query': 'ما هي آخر الأخبار السياسية؟',
 'context': 'ريما مكتبي مع جنبلاط الفراغ الرئاسي في لبنان والاتصالات الجارية للتوصل الى انتخاب رئيس للجمهورية. كما تتناول المقابلة الوضع في سوريا والتوقعات بالنسبة للمرحلة المقبلة، والتاثير على لبنان، اضافة الى ما حمله الى بيروت وزير الخارجية الاميركية جون كيري في زيارته الاخيرة. كما تتطرق المقابلة للتدهور التي ترفعها الحركة هي رفض صمت الرئيس والحكومة والقوى السياسية عما يحدث على الارض. رفض سيطرة الميليشيات المسلحة على العاصمة وعلى بقية المحافظات. رفض الاتفاقيات السياسية مع الميليشيات المسلحة ما لم تفض الى نزع السلاح. رفض اقتحام معسكرات الجيش والامن ونهب السلاح. رفض دمج الميليشيات المسلحة في الجيش جنبلاط ضيف حصري على قناة الحدث الثلاثاء الحدث.نت يحل رئيس الحزب الاشتراكي اللبناني وليد جنبلاط غدا الثلاثاء ضيفا على قناة الحدث، في مقابلة حصرية تبث الساعة الخامسة عصرا بتوقيت بيروت، الثانية بعد الظهر بتوقيت غرينتش. وتتناول المقابلة التي تجريها الزميلة ريما مكتبي مع جنبلاط الفراغ الرئاسي في لبنان رسالة سرعان ما اشعلت المعارض

In [12]:
print(search_news("الوضع في مصر"))
print(summarize_topic("الوضع في سوريا"))
print(compare_timeline("الانتخابات اللبنانية", category="Politics"))
print(answer_direct("ما هو التضخم؟"))

{'tool': 'search_news', 'query': 'الوضع في مصر', 'results': [{'id': 'ec9bdb092b9d4d27dbb735e2f0c75b6f', 'category': 'Politics', 'text': 'بناء الدولة السورية وكذلك حزب الاتحاد الديموقراطي الكردي في سوريا .', 'score': 0.5}, {'id': 'cb4bb2bc53ab4c761cdfb1dc7f7215ac', 'category': 'Politics', 'text': 'جلسة طارئة اليوم بطلب من فرنسا وبريطانيا لبحث الوضع في حلب. وسيستمع اعضاء المجلس الى احاطة بشان الوضع في شرق حلب من احد مسؤولي الشؤون الانسانية في الامم المتحدة، وكذلك من المبعوث الدولي الى سوريا ستيفان دي ميستورا. من جانبه، افاد مراسل الحدث في نيويورك بان مشروع قرار ثلاثيا قدمته مصر واسبانيا', 'score': 0.5}, {'id': '6ff995897c2be735ede45ec1bb535b5f', 'category': 'Politics', 'text': 'السعودية هو جريمة مضافة وليس حلا. نقلا عن العرب جميع المقالات المنشورة تمثل راي كتابها فقط.', 'score': 0.3333333333333333}, {'id': '182bed0cee7183d5900ba61d2ecdaf69', 'category': 'Politics', 'text': 'الوفد الافريقي برئاسة الفا عمر كوناري، الرئيس الاسبق لمفوضية الاتحاد الافريقي، تعد الزيارة الثالثة للوفد الى مصر لت

In [13]:
search_news("الاقتصاد")

{'tool': 'search_news',
 'query': 'الاقتصاد',
 'results': [{'id': 'ff8b08c3e675f3f71e0e6a2cfab645d3',
   'category': 'Politics',
   'text': 'موعد الاجتماع لاحقا. بيان الشيخ عبدالله بن علي ال ثاني',
   'score': 0.5},
  {'id': '8f7efbcf4caae6050e8ef3944f65fb49',
   'category': 'Medical',
   'text': 'الاقتصاد بفارق نقطة كاملة. وهذا يدل على وجود تحول عن انتخابات عام ، حين كان الاقتصاد في صدارة اهتمامات الناخبين وجاءت الصحة في المركز الثاني. ويعلم كاميرون ان موقفه ليس قويا في هذه القضية لان حزب العمال المعارض الذي ينافسه هو الذي انشا الهيئة.',
   'score': 0.5},
  {'id': '9918b9f2d988c7b8fa42927da61b98d8',
   'category': 'Medical',
   'text': 'العامين و، فيما ارتفع عدد السكان خلال هذه الفترة بنسبة . فقط.',
   'score': 0.3333333333333333},
  {'id': '2fa2f101563d01544b57744f87c9a77a',
   'category': 'Politics',
   'text': 'المسار الذي رسمه المرشد عليخامنئي تحت ما سماه الاقتصاد المقاوم توفير فرص العمل والانتاج صعوبة اضافية امام تعافي الاقتصاد. وتكبر الخشية في ايران من ان يتحول الرهان على نهضة ا

In [15]:
print(client.get_collection("arabic_news").points_count)

26320
